<img src="images/highway2.jpg" width="90%">

# Predictive Analytics & Forecasting
> Executive Master in Business Analytics and AI
> - Project 2026
> - Rita P. Ribeiro - University of Porto | INESC TEC
> - October 2026


## Description
You are part of an analytics team supporting a national road-safety authority. The organization wants to understand how historical highway-accident data can be used to support better road-safety decisions.

# The modeling data

`model_table.csv` covers **1 April 2018 (`ERA_START`) to 31 December 2025**: one row per region per day, with the target (`accidents`) and all features. Training, validation and test are **date ranges of this one table**, split by time, never randomly.

### Training rows start in 2019Q3

The history features need a full year of data before each quarter's cutoff. For the quarters 2018Q2 to 2019Q2 that year is not complete yet, so some history columns are empty:

| Quarter | Cutoff | `hist_avg_365d` needs data from… | Available? |
|---|---|---|---|
| 2019Q2 | 1 Mar 2019 | 1 Mar 2018 | no: the data starts 1 Apr 2018, so the value is empty |
| **2019Q3** | 1 Jun 2019 | 1 Jun 2018 | **yes: the first complete quarter** |

- **The 2018Q2–2019Q2 data is still used:** the history features of later quarters are calculated from it (for example, `hist_avg_365d` for 2019Q3 is the average from June 2018 to May 2019). Those days are just not training *rows*.
- **Why not train from 2018Q2?** The GLM cannot handle empty values. Starting all three models at 2019Q3 means they train on exactly the same rows, so the comparison is fair. We give up 2,280 of 13,625 rows (about 17%).

### Training, validation and test

| Part | Period | Rows (region × day) | Used for |
|---|---|---|---|
| **Training** | 2019Q3 up to each quarter's cutoff | 5,865 before 2023, growing each quarter | fitting the models |
| **Validation** | 2023Q1 – 2024Q4 (8 quarters) | 3,655 (1,170 weekend/holiday) | comparing models, tuning, ensemble weights |
| **Test** | 2025Q1 – 2025Q4 (4 quarters) | 1,825 (585 weekend/holiday) | the final score, used once |

- **Rolling origin:** each quarter is predicted by a model trained only on the days before its cutoff (1 month before the quarter starts). The month between the cutoff and the quarter start is never used, because in real use it has not happened yet.
- **COVID days** (16 March – 30 June 2020) are left out of training.
- **All days are used for training.** The main score is measured on weekends and holidays (the days the planners use), with the all-days score as an extra.

# Task 3 - Modelling

Forecast the number of accidents with victims per region per day, for every weekend day and holiday of the next quarter, 1 month before the quarter starts.

**Contents**
1. Setup
2. How we evaluate
3. Baseline
4. Model 1 - GLM Poisson regression
5. Model 2 - XGBoost
6. Model 3 - Random Forest
7. Comparison
8. Ensemble
9. Final test and interpretation
10. Conclusions and limitations

--------
## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_poisson_deviance
from xgboost import XGBRegressor

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "data/processed"

# Shared calendar and region functions (src/features.py)
sys.path.append(str(ROOT / "src"))
from features import REGIONS, REGION_EN, build_calendar, get_brazil_holidays, assign_region, haversine_km

# model_table.csv is created in 02_Data_Understanding_Preparation.ipynb, section 6.4
model_table = pd.read_csv(PROCESSED / "model_table.csv", parse_dates=["date"])
model_table.head()

# ---- decisions agreed in Task 1 ----
LEAD_MONTHS = 1         # data cutoff (ref_date) = 1 month before the quarter starts
WINDOW_DAYS = 3         # holiday window = holiday date +/- 3 days
QUARTER_DAYS = 91       # "last quarter" = the 91 days up to the cutoff
ERA_START = pd.Timestamp("2018-04-01")   # start of the modelling period
# Target = accidents with victims = injury (1) or fatal (2) per region per day. Class 0 is excluded. 

REGIONS = ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]

# plotting: restrained palette, light grid, no top/right spines
BLUE, ORANGE, GREY, DARK = "#2F6DB5", "#D9622B", "#8A8F98", "#222222"
REGION_COLORS = dict(zip(REGIONS, [BLUE, ORANGE, "#1B998B", "#7A5FB5", "#C79A1E"]))
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#E3E5E8", "grid.linewidth": 0.7, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": DARK, "text.color": DARK, "xtick.color": DARK, "ytick.color": DARK,
    "legend.frameon": False, "figure.figsize": (8, 3.6),
})
pd.set_option("display.max_columns", 50, "display.width", 140, "display.float_format", "{:,.3f}".format)

--------
## 2. How we evaluate

- **Split by time, never randomly.**
- **Rolling backtest:** for each past quarter, train on data up to 1 month before the quarter starts, then predict that quarter's weekends and holidays.
- The **last quarters are kept as a final test** and are only used in section 9.
- **Metrics:** MAE, RMSE and Poisson deviance.
- The backtest periods are defined **once** here and reused by every model, so the ensemble can combine predictions for exactly the same days.

Decision:
which days to train on. Our forecast only weekends and holidays, which are 908 of the 2,832 days. I'd suggest:

* Train on all days, with is_weekend and is_holiday as features. 
* That's three times more data, and weekdays still teach the model each region's level and trend.
* Evaluate only on weekends and holidays, because those are the days the planners use.

In [ ]:
# TODO: define the backtest periods (train end, cutoff, forecast quarter) and the final test quarters

--------
## 3. Baseline

A simple rule to beat, for example the same day type's average in the previous year. A model is only worth using if it does better than this.

In [ ]:
# TODO: baseline predictions on the backtest periods

--------
## 4. Model 1 - GLM Poisson regression

Check whether the counts are more spread out than a Poisson model allows (overdispersion). If they are, also try a negative binomial model.

In [ ]:
# TODO: GLM Poisson (statsmodels) on the backtest periods

--------
## 5. Model 2 - XGBoost

`XGBRegressor(objective="count:poisson")`

In [ ]:
# TODO: XGBoost on the backtest periods

--------
## 6. Model 3 - Random Forest

`RandomForestRegressor(criterion="poisson")`

In [ ]:
# TODO: Random Forest on the backtest periods

--------
## 7. Comparison

All three models and the baseline on the same backtest periods, overall and per region.

In [ ]:
# TODO: metrics table and plots

--------
## 8. Ensemble

Average the three models' predictions. Any weights are chosen on the backtest periods only, never on the final test.

In [ ]:
# TODO: ensemble predictions and metrics

--------
## 9. Final test and interpretation

Run the chosen models once on the final test quarters. Which features matter, and where does the model fail?

In [ ]:
# TODO: final test metrics, feature importance, errors by region and day type

--------
## 10. Conclusions and limitations